# Verification: Bivalent Hapten + Monovalent Inhibitor

Compares NFsim against extended kinetic ODEs that include monovalent
hapten competition. The monovalent hapten B has only 1 binding site,
so no factor-of-2 applies to its capture term (unlike the bivalent L
which has 2 identical r sites).

Three bivalent doses tested with fixed monovalent hapten at 3e7/cell.

In [ ]:
import subprocess, os, glob
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.')

NA = 6.02214076e23; V_cell = 1e-9; R_per_cell = 3e5; f = 0.01
V_sim = V_cell * f; RT = R_per_cell * f; ST = 2 * RT
kon = 1e6; koff = 0.01; KxRT = 5
kf = kon / (NA * V_sim); kxf = KxRT / RT * koff
kf_mono = kf  # same single-site affinity

LT_values = [3e5, 3e6, 3e7]
LT_labels = ['below', 'optimal', 'above']
BT_per_cell = 3e7
t_end = 3000; n_steps = 300
print(f'RT={RT:.0f}, ST={ST:.0f}, BT/cell={BT_per_cell:.0e}')

## Extended ODEs with Monovalent Competitor

$$\frac{dm}{dt} = 2 k_f L S - k_{\text{off}} m - k_{xf} m S + 2 k_{\text{off}} M$$
$$\frac{dM}{dt} = k_{xf} m S - 2 k_{\text{off}} M$$
$$\frac{dm_B}{dt} = k_f^{\text{mono}} B_{\text{free}} S - k_{\text{off}} m_B$$

Conservation: $S = S_0 - m - 2M - m_B$. Note: bivalent capture
has factor 2 (2 r sites), monovalent does NOT (1 r site).

In [ ]:
def odes_with_mono(t, y, LT_sim, BT_sim, kf_r, kxf_r, kf_mono_r,
                   koff_r, ST_r):
    m, M, mB = y
    S = max(ST_r - m - 2*M - mB, 0)
    L = max(LT_sim - m - M, 0)
    B = max(BT_sim - mB, 0)
    # Factor of 2 for bivalent ligand (2 r sites)
    dm = 2*kf_r*L*S - koff_r*m - kxf_r*m*S + 2*koff_r*M
    dM = kxf_r*m*S - 2*koff_r*M
    # No factor of 2 for monovalent (1 r site)
    dmB = kf_mono_r*B*S - koff_r*mB
    return [dm, dM, dmB]

ode_results = {}
for LT_val, label in zip(LT_values, LT_labels):
    LT_sim = LT_val * f; BT_sim = BT_per_cell * f
    sol = solve_ivp(odes_with_mono, (0, t_end), [0, 0, 0],
                    args=(LT_sim, BT_sim, kf, kxf, kf_mono, koff, ST),
                    method='LSODA', rtol=1e-8, atol=1e-10,
                    t_eval=np.linspace(0, t_end, n_steps+1))
    m, M, mB = sol.y[0], sol.y[1], sol.y[2]
    BLR = m + 2*M; S = ST - m - 2*M - mB
    ode_results[label] = dict(t=sol.t, m=m, M=M, mB=mB,
                              Bonds_LR=BLR, Bonds_BR=mB,
                              S=S, LT_sim=LT_sim)
    print(f'{label}: m={m[-1]:.1f}, M={M[-1]:.1f}, '
          f'mB={mB[-1]:.1f}, BLR={BLR[-1]:.1f}')

## Run NFsim

In [ ]:
bngl_file = ('symmetric_bivalent_hapten_receptor_crosslinking'
             '_dembo1978_monovalent_inhibitor.bngl')
with open(bngl_file) as fh:
    bngl_template = fh.read()

nf = {}
f_overrides = {'below': 0.01, 'optimal': 0.003, 'above': 0.002}
for LT_val, label in zip(LT_values, LT_labels):
    f_run = f_overrides[label]
    tmp = bngl_template.replace('LT_per_cell  3e6',
                                f'LT_per_cell  {LT_val:.0e}')
    tmp = tmp.replace('BT_per_cell  3e7',
                      f'BT_per_cell  {BT_per_cell:.0e}')
    if f_run != f:
        tmp = tmp.replace('f  0.01  # dimensionless',
                          f'f  {f_run}  # dimensionless')
    tmp = tmp.replace('suffix=>"nfr"', f'suffix=>"nfr_{label}"')
    tmp_file = f'tmp_mono_{label}.bngl'
    with open(tmp_file, 'w') as fh: fh.write(tmp)
    RT_run = R_per_cell * f_run; LT_run = LT_val * f_run
    BT_run = BT_per_cell * f_run
    print(f'Running {label} (f={f_run}, RT={RT_run:.0f}, '
          f'LT={LT_run:.0f}, BT={BT_run:.0f})...',
          end=' ', flush=True)
    subprocess.run(['bionetgen', 'run', '-i', tmp_file],
                   capture_output=True, text=True, timeout=600)
    gdat = f'tmp_mono_{label}_nfr_{label}.gdat'
    if os.path.exists(gdat):
        d = np.loadtxt(gdat, comments='#')
        nf[label] = dict(t=d[:,0], Free_L=d[:,1], Free_B=d[:,2],
                         Free_R=d[:,3], Bonds_LR=d[:,4],
                         Bonds_BR=d[:,5], Free_R_sites=d[:,6],
                         LT_sim=LT_run, BT_sim=BT_run,
                         RT_run=RT_run, f_run=f_run)
        print(f'{len(d)} pts, BLR~{d[-60:,4].mean():.0f}, '
              f'BBR~{d[-60:,5].mean():.0f}')
    else: print('FAILED')
for fn in glob.glob('tmp_mono_*'): os.remove(fn)

## Comparison Plots

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
colors = {'below': 'C0', 'optimal': 'C1', 'above': 'C2'}
for col, label in enumerate(LT_labels):
    if label not in nf:
        for row in range(3):
            axes[row, col].text(0.5, 0.5, 'No data',
                transform=axes[row, col].transAxes, ha='center')
        continue
    ode = ode_results[label]; r = nf[label]
    RT_r = r['RT_run']; ST_r = 2*RT_r
    ax = axes[0, col]
    ax.plot(ode['t'], ode['Bonds_LR']/ST, 'k-', lw=2, label='ODE')
    ax.plot(r['t'], r['Bonds_LR']/ST_r, '-', color=colors[label],
            lw=1, alpha=0.7, label='NFsim')
    ax.set_ylabel('LR Bonds / ST' if col==0 else '')
    ax.set_title(f'{label}', fontsize=10); ax.legend(fontsize=8)
    ax = axes[1, col]
    ax.plot(ode['t'], ode['Bonds_BR']/ST, 'k-', lw=2, label='ODE')
    ax.plot(r['t'], r['Bonds_BR']/ST_r, '-', color=colors[label],
            lw=1, alpha=0.7, label='NFsim')
    ax.set_ylabel('BR Bonds / ST' if col==0 else '')
    ax.legend(fontsize=8)
    ax = axes[2, col]
    ax.plot(ode['t'], ode['S']/ST, 'k-', lw=2, label='ODE')
    ax.plot(r['t'], r['Free_R_sites']/ST_r, '-', color=colors[label],
            lw=1, alpha=0.7, label='NFsim')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Free R sites / ST' if col==0 else '')
    ax.legend(fontsize=8)
fig.suptitle(f'Extended ODEs (monovalent inhibitor, BT/cell={BT_per_cell:.0e}) '
             'vs NFsim\nBlack = ODE, colored = NFsim', fontsize=12)
fig.tight_layout()
fig.savefig('verify_dembo1978_monovalent.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved verify_dembo1978_monovalent.png')

## Error Summary

In [ ]:
print('=== Equilibrium comparison (normalized) ===')
print(f'{"":>10s} {"BLR/ST":>16s} {"BBR/ST":>16s}')
print(f'{"Dose":>10s} {"ODE":>7s} {"NFsim":>7s} {"ODE":>7s} {"NFsim":>7s}')
print('-' * 50)
max_rel_err = 0
for label in LT_labels:
    if label not in nf: continue
    ode = ode_results[label]; r = nf[label]; n_eq = 60
    RT_r = r['RT_run']; ST_r = 2*RT_r
    BLR_nf = r['Bonds_LR'][-n_eq:].mean()/ST_r
    BBR_nf = r['Bonds_BR'][-n_eq:].mean()/ST_r
    BLR_ode = ode['Bonds_LR'][-1]/ST
    BBR_ode = ode['Bonds_BR'][-1]/ST
    err_LR = abs(BLR_nf-BLR_ode)/max(BLR_ode,1e-10)
    err_BR = abs(BBR_nf-BBR_ode)/max(BBR_ode,1e-10)
    max_rel_err = max(max_rel_err, err_LR, err_BR)
    print(f'{label:>10s} {BLR_ode:7.4f} {BLR_nf:7.4f} '
          f'{BBR_ode:7.4f} {BBR_nf:7.4f}')
print(f'\nMax relative error: {max_rel_err:.4f}')
if max_rel_err < 0.10:
    print('PASS: NFsim monovalent inhibitor agrees with extended ODEs.')
else:
    print('FAIL: Significant disagreement detected.')